# EDA

## Daten laden
europe-electricity-load-hourly-20192025 aus Kaggle manuell herunterladen und als csv einlesen

In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub
import shutil
import os

dataset_link = "dsersun/europe-electricity-load-hourly-20192025"  # just the owner/dataset part
destination = "../data/raw"

cache_path = kagglehub.dataset_download(dataset_link)
#print(f"Downloaded to cache: {cache_path}")

os.makedirs(destination, exist_ok=True)

# Copy all files from cache to destination
for file in os.listdir(cache_path):
    shutil.copy(os.path.join(cache_path, file), destination)
    print(f"Copied: {file} → {destination}")

In [ ]:
import pandas as pd

# Load the dataset
file_path = "../data/raw/MHLV_2019_2025_combined.csv"
df_energy = pd.read_csv(file_path, parse_dates=['DateUTC', 'DateShort'])

display(df_energy.head())

print(df_energy.info())

print(df_energy.describe())

In [ ]:
print('MeasureItem unique values:')
print(df_energy['MeasureItem'].unique())

print('Cov_ratio unique values:')
print(df_energy['Cov_ratio'].unique())

print('CountryCode value counts:')
print(df_energy['CountryCode'].value_counts())

print('Compare Value and Value_ScaleTo100 columns:')
print(df_energy[df_energy['Value']!=df_energy[ 'Value_ScaleTo100']].head())

In [ ]:
df_energy = df_energy.drop(columns=['DateShort', 'MeasureItem', 'TimeFrom', 'TimeTo', 'Cov_ratio', 'Value_ScaleTo100'], axis=1)
df_energy = df_energy.rename(columns={'Value': 'EnergyDemand'})
df_energy.head()


In [ ]:
df_energy.isna().sum()

In [ ]:
df_energy['hour'] = df_energy['DateUTC'].dt.hour
df_energy['weekday'] = df_energy['DateUTC'].dt.dayofweek
df_energy['month'] = df_energy['DateUTC'].dt.month
df_energy['is_weekend'] = (df_energy['DateUTC'].dt.weekday >= 5).astype(int)
df_energy.head()

In [ ]:
df_energy_de = df_energy[df_energy['CountryCode'] == 'DE']
df_energy_de = df_energy_de.drop(columns=['CountryCode'], axis=1)

df_energy_de.to_csv("../data/energy_demand_de_2019_2025.csv", index=False)

In [ ]:
%pip install holidays

In [ ]:
# add public holidays for Germany
import pandas as pd
import holidays 

df_energy_de = pd.read_csv("../data/energy_demand_de_2019_2025.csv", parse_dates=['DateUTC'])
de_holidays = holidays.Germany(years=range(2019, 2026))
df_energy_de['is_holiday'] = df_energy_de['DateUTC'].dt.date.apply(lambda x: 1 if x in de_holidays else 0)

# add holiday ratio depending the number of states in Germany with a holiday on that day
def holiday_ratio(date):
    count = sum([1 for state in holidays.Germany(years=[date.year]).items() if state[0] == date])
    return count / 16   
df_energy_de['holiday_ratio'] = df_energy_de['DateUTC'].dt.date.apply(holiday_ratio)

In [ ]:
df_energy_de.to_csv("../data/energy_demand_de_2019_2025.csv", index=False)

In [ ]:
df_energy_de.head()

In [ ]:
df_energy_de.describe()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd   

df_energy_de = pd.read_csv("../data/energy_demand_de_2019_2025.csv", parse_dates=['DateUTC'])

df_monthly = df_energy_de.set_index('DateUTC')['EnergyDemand'].resample('ME').mean().reset_index()

# Convert dates to float for regression
df_monthly['DateNum'] = mdates.date2num(df_monthly['DateUTC'])

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=df_monthly, x='DateUTC', y='EnergyDemand', ax=ax)
plt.title('Electricity Load in Germany (2019-2025) - Monthly Mean')

# add trend line
sns.regplot(data=df_monthly, x='DateNum', y='EnergyDemand',
            scatter=False, color='red', label='Trend Line', ax=ax)

plt.ylim(0, df_monthly['EnergyDemand'].max() * 1.1)
plt.grid()
plt.xlabel('Year')
plt.ylabel('Load (MW)')
plt.show()

In [ ]:
df_weekly = df_energy_de.set_index('DateUTC')['EnergyDemand'].resample('W').mean().reset_index()
# Convert dates to float for regression
df_weekly['DateNum'] = mdates.date2num(df_weekly['DateUTC'])

fig, ax = plt.subplots(figsize=(16, 6))
sns.lineplot(data=df_weekly, x='DateUTC', y='EnergyDemand', ax=ax)
plt.title('Electricity Load in Germany (2019-2025) - Weekly Mean')

# add trend line
sns.regplot(data=df_weekly, x='DateNum', y='EnergyDemand',
            scatter=False, color='red', label='Trend Line', ax=ax)

plt.ylim(0, df_weekly['EnergyDemand'].max() * 1.1)
plt.grid()
plt.xlabel('Year')
plt.ylabel('Load (MW)')
plt.show()

* additive model - seasonal values and redisual independent on trend

### time series decomposition

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_energy_de = pd.read_csv("../data/energy_demand_de_2019_2025.csv", parse_dates=['DateUTC'])
df_monthly = df_energy_de.set_index('DateUTC')['EnergyDemand'].resample('ME').mean().reset_index()
df_monthly.set_index('DateUTC', inplace=True)
# Convert dates to float for regression
df_monthly['DateNum'] = mdates.date2num(df_monthly.index)

slope, intercept = np.polyfit(df_monthly['DateNum'], df_monthly['EnergyDemand'], 1) # estimate line coefficient
trend = df_monthly['DateNum'] * slope + intercept # linear trend
detrended = df_monthly['EnergyDemand'] - trend # remove the trend

plt.figure(figsize=(10, 5))
plt.plot(df_monthly['EnergyDemand'], label='Original')
plt.plot(trend, label='Trend')
plt.plot(detrended, label='Detrended')

plt.grid()
plt.legend()
plt.show()

In [ ]:
!pip install statsmodels

In [ ]:
# decompose seasonality using seasonal_decompose
from statsmodels.tsa.seasonal import seasonal_decompose     
#df_monthly.set_index('DateUTC', inplace=True)
decomposition = seasonal_decompose(df_monthly['EnergyDemand'], model='additive', period=12) # monthly data, so period is 12 months
decomposition.plot();  # use ; to avoid doubled plot. the returned figure is already plotted by the function, so we just suppress the output to avoid a second empty plot.

In [ ]:
# decompose seasonality using seasonal_decompose with multiplicative model
from statsmodels.tsa.seasonal import seasonal_decompose     
#df_monthly.set_index('DateUTC', inplace=True)
decomposition = seasonal_decompose(df_monthly['EnergyDemand'], model='multiplicative', period=12) # monthly data, so period is 12 months
decomposition.plot();  # use ; to avoid doubled plot. the returned figure is already plotted by the function, so we just suppress the output to avoid a second empty plot.
decomposition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from scipy.fft import fft
np.random.seed(0)  # for reproducibility

stl_decomposition = STL(endog=df_monthly['EnergyDemand'], period=12, robust=True).fit()
stl_decomposition.plot();  # use ; to avoid doubled plot. the returned figure is already plotted by the function, so we just suppress the output to avoid a second empty plot.